## CANDIDATE RANKING

In [10]:
import scipy.sparse as sp
import numpy as np
import pandas as pd

In [2]:
import joblib

# Load best model (you saved earlier)
best_model = joblib.load("../models/best_model.pkl")

print("✅ Model loaded!")

✅ Model loaded!


In [ ]:


X_test = sp.load_npz("../data/X_test.npz")
df_test = pd.read_csv("../data/resume_jd_test_cleaned.csv")
bert_resume_test = np.load("../data/test_resume_embeddings.npy")
bert_jd_test = np.load("../data/test_jd_embeddings.npy")

bert_test = np.hstack([bert_resume_test, bert_jd_test])
print("✅ Test data loaded!")
print(X_test.shape)

✅ Test data loaded!
(1759, 10001)


In [7]:
import joblib

le = joblib.load("../models/label_encoder.pkl")

print("Classes:", le.classes_)

Classes: ['Good Fit' 'No Fit' 'Potential Fit']


### 🔹 Ranking Factors

We use three main components:

1. **Confidence Score**
   - Obtained from model prediction probabilities
   - Indicates how certain the model is about its prediction

2. **Cosine Similarity**
   - Based on TF-IDF vectors
   - Measures keyword-level similarity between resume and job description

3. **BERT Similarity**
   - Based on semantic embeddings
   - Captures contextual meaning between texts

In [8]:
# Combine resume + JD embeddings
bert_test = np.hstack([bert_resume_test, bert_jd_test])

print("Combined BERT shape:", bert_test.shape)

Combined BERT shape: (1759, 768)


In [12]:
from scipy.sparse import hstack
X_test = hstack([X_test, bert_test])

print("Final X_test shape:", X_test.shape)

Final X_test shape: (1759, 10769)


In [13]:
y_pred = best_model.predict(X_test)
proba = best_model.predict_proba(X_test)

print("✅ Predictions done!")

✅ Predictions done!


In [14]:
df_ranking = df_test.copy()

# Convert numeric labels → original labels
df_ranking['predicted_label'] = le.inverse_transform(y_pred)

# Confidence score (highest probability)
df_ranking['confidence'] = proba.max(axis=1)

df_ranking.head()

,resume_text,job_description_text,label,resume_clean,jd_clean,cosine_similarity,bert_similarity,predicted_label,confidence
0,Summary7+ years of experience as a BI develope...,Key Responsibilities:Create intricate wiring n...,No Fit,summary7 year experience developer proven trac...,key responsibility create intricate wiring net...,0.035738,0.564143,No Fit,0.969750
1,Professional BackgroundAnalyst versed in data ...,Personal development and becoming the best you...,No Fit,professional backgroundanalyst versed data ana...,personal development becoming best growth expl...,0.100394,0.631430,No Fit,0.964767
2,Executive ProfileDedicated professional with t...,"Location: Tampa, FL\r\nExp: 7-10 Yrs\r\nSPOC: ...",No Fit,executive profilededicated professional accomp...,location tampa exp yr spoc tushar kshirsagar k...,0.045090,0.631957,No Fit,0.976654
3,"Summarytyee\r\nHighlightsMicrosoft Excel, Word...","Primary Location: Melbourne, Florida\r\nV-Soft...",No Fit,summarytyee highlightsmicrosoft excel word out...,primary location melbourne florida soft consul...,0.050667,0.465062,No Fit,0.808689
4,SummaryEIT certified Engineer and ASTQB Certif...,At Oregon Specialty Group the Accounting & Pay...,No Fit,summaryeit certified engineer astqb certified ...,oregon specialty group accounting payroll depa...,0.082104,0.645818,No Fit,0.912367


## sample example

In [15]:
df_ranking.loc[18, [
    'predicted_label',
    'confidence',
    'cosine_similarity',
    'bert_similarity'
]]

predicted_label      Potential Fit
confidence                0.829392
cosine_similarity         0.195399
bert_similarity           0.784537
Name: 18, dtype: object

In [16]:
df_ranking['final_score'] = (
    0.5 * df_ranking['confidence'] +
    0.3 * df_ranking['bert_similarity'] +
    0.2 * df_ranking['cosine_similarity']
)

In [17]:
def assign_tier(label):
    if label == 'Good Fit':
        return 1
    elif label == 'Potential Fit':
        return 2
    else:
        return 3

df_ranking['tier'] = df_ranking['predicted_label'].apply(assign_tier)

In [18]:
df_ranking = df_ranking.sort_values(
    ['tier', 'final_score'],
    ascending=[True, False]
).reset_index(drop=True)

df_ranking['rank'] = df_ranking.index + 1

In [19]:
df_ranking[[
    'rank',
    'predicted_label',
    'confidence',
    'cosine_similarity',
    'bert_similarity',
    'final_score'
]].head(10)

,rank,predicted_label,confidence,cosine_similarity,bert_similarity,final_score
0,1,Good Fit,0.953331,0.111224,0.746746,0.722934
1,2,Good Fit,0.940114,0.133801,0.709332,0.709617
2,3,Good Fit,0.851130,0.160198,0.833031,0.707514
3,4,Good Fit,0.917275,0.124725,0.703598,0.694662
4,5,Good Fit,0.931140,0.068933,0.695878,0.688120
5,6,Good Fit,0.888942,0.144837,0.702125,0.684076
6,7,Good Fit,0.769774,0.269229,0.812305,0.682424
7,8,Good Fit,0.872734,0.182528,0.695470,0.681514
8,9,Good Fit,0.916010,0.089216,0.681749,0.680373
9,10,Good Fit,0.884777,0.085847,0.735285,0.680143


### 🔹 Final Scoring Formula

The final ranking score is calculated as:

#### Explanation:
- **0.5 (Confidence):** Most important, reflects model certainty  
- **0.3 (BERT):** Captures semantic meaning (very important)  
- **0.2 (Cosine):** Captures keyword overlap (less important)

---

### 🔹 Tier-Based Ranking Logic

Candidates are categorized into tiers based on predicted labels:

| Label          | Tier | Meaning              |
|---------------|------|----------------------|
| Good Fit      | 1    | Highly suitable      |
| Potential Fit | 2    | Moderately suitable  |
| No Fit        | 3    | Not suitable         |



# TOP 10

In [20]:
df_ranking[[
    'rank',
    'predicted_label',
    'confidence',
    'cosine_similarity',
    'bert_similarity',
    'final_score'
]].head(10)

,rank,predicted_label,confidence,cosine_similarity,bert_similarity,final_score
0,1,Good Fit,0.953331,0.111224,0.746746,0.722934
1,2,Good Fit,0.940114,0.133801,0.709332,0.709617
2,3,Good Fit,0.851130,0.160198,0.833031,0.707514
3,4,Good Fit,0.917275,0.124725,0.703598,0.694662
4,5,Good Fit,0.931140,0.068933,0.695878,0.688120
5,6,Good Fit,0.888942,0.144837,0.702125,0.684076
6,7,Good Fit,0.769774,0.269229,0.812305,0.682424
7,8,Good Fit,0.872734,0.182528,0.695470,0.681514
8,9,Good Fit,0.916010,0.089216,0.681749,0.680373
9,10,Good Fit,0.884777,0.085847,0.735285,0.680143


In [21]:
# BOTTOM 10

In [22]:
df_ranking[[
    'rank',
    'predicted_label',
    'confidence',
    'cosine_similarity',
    'bert_similarity',
    'final_score'
]].tail(10)

,rank,predicted_label,confidence,cosine_similarity,bert_similarity,final_score
1749,1750,No Fit,0.407771,0.005033,0.579447,0.378726
1750,1751,No Fit,0.536607,0.018763,0.316570,0.367027
1751,1752,No Fit,0.459920,0.051890,0.418822,0.365985
1752,1753,No Fit,0.488858,0.085053,0.335031,0.361949
1753,1754,No Fit,0.494601,0.015310,0.371579,0.361836
1754,1755,No Fit,0.514885,0.027727,0.304737,0.354409
1755,1756,No Fit,0.575440,0.027869,0.194295,0.351582
1756,1757,No Fit,0.503019,0.012768,0.321230,0.350432
1757,1758,No Fit,0.395624,0.052997,0.390431,0.325540
1758,1759,No Fit,0.401390,0.033278,0.350799,0.312590


### 🔹 Conclusion

This ranking approach ensures:
- Better prioritization of candidates
- Combination of model prediction and semantic understanding
- Improved accuracy in resume-job matching